In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma


docs = [
    "LLM은 이전 토큰들을 보고 다음 토큰의 확률을 예측하는 모델이다.",
    "임베딩은 텍스트의 의미를 벡터로 바꾸며, 의미가 비슷할수록 벡터가 가깝다.",
    "RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.",
    "벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.",
    "파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.",
    "프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.",
]
topics = ["LLM", "임베딩", "RAG", "벡터DB", "파인튜닝", "프롬프트"]

documents = [Document(page_content=text, metadata={"topic": topic}) for text, topic in zip(docs, topics)]
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = Chroma.from_documents(documents, embedding=embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Metadata filtering

In [2]:
# 1) 필터 없이
results = vectorstore.similarity_search("모델 가중치를 학습", k=3)
for r in results:
    print(r.metadata["topic"], "|", r.page_content)



파인튜닝 | 파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.
LLM | LLM은 이전 토큰들을 보고 다음 토큰의 확률을 예측하는 모델이다.
RAG | RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.


In [3]:
# 2) 필터 있이
results = vectorstore.similarity_search("모델 가중치를 학습", k=3, filter={"topic": "프롬프트"})
for r in results:
    print(r.metadata["topic"], "|", r.page_content)

프롬프트 | 프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.


HyBrid Search

In [7]:
from langchain_community.retrievers import BM25Retriever      # 이건 그대로 (됐잖아)
from langchain_classic.retrievers import EnsembleRetriever     # ← langchain → langchain_classic

bm25 = BM25Retriever.from_documents(documents)        # sparse (키워드)
bm25.k = 3

dense = vectorstore.as_retriever(search_kwargs={"k": 3})   # 기존 벡터

ensemble = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])
results = ensemble.invoke("질문")
results

[Document(metadata={'topic': '프롬프트'}, page_content='프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.'),
 Document(metadata={'topic': '파인튜닝'}, page_content='파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.'),
 Document(id='c9d8f6d4-2de9-4810-b544-b9b788434001', metadata={'topic': 'RAG'}, page_content='RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.'),
 Document(metadata={'topic': '벡터DB'}, page_content='벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.')]

reranking

In [8]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever

model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")   # 다국어 reranker
reranker = CrossEncoderReranker(model=model, top_n=3)

base = vectorstore.as_retriever(search_kwargs={"k": 10})   # 1차: 많이 뽑고
compression = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=base          # 2차: 정밀 재정렬
)
results = compression.invoke("질문")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

c:\Users\조영석\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\조영석\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [9]:
results

[Document(id='a63a5c56-511d-4f5a-8632-c7f38ecc2b0a', metadata={'topic': '벡터DB'}, page_content='벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.'),
 Document(id='77c2024f-928b-4a40-b124-7b72fde971c5', metadata={'topic': 'LLM'}, page_content='LLM은 이전 토큰들을 보고 다음 토큰의 확률을 예측하는 모델이다.'),
 Document(id='548d89bf-66ac-4b8c-8e3a-1c56856a25b7', metadata={'topic': '임베딩'}, page_content='임베딩은 텍스트의 의미를 벡터로 바꾸며, 의미가 비슷할수록 벡터가 가깝다.')]